# Only understand the syntax tree and the pseudocode.

- Understand the code  especially the output itself possible  patterns( 1 hrs )
- Come up with 3 experiments to improve it using the subsample approach itself and 2 experiments to run and find what interventions could be done to figure out the issue (1 hr)
- Implement figure things out ( 3 hrs )
Goal - Atleast get a 30% improvement on current AUROC of 0.61

In [176]:
import os, re

def explore(path, max_show=3, max_depth=4):
    def walk(p, depth=0):
        if depth > max_depth:
            return
        try:
            entries = sorted(os.listdir(p))
        except (PermissionError, NotADirectoryError):
            return
        entries = [e for e in entries if not e.startswith('.')]
        dirs  = [e for e in entries if os.path.isdir(os.path.join(p, e))]
        files = [e for e in entries if not os.path.isdir(os.path.join(p, e))]
        ind = '  ' * depth

        # group dirs by their name-pattern (digits -> #)
        groups = {}
        for d in dirs:
            groups.setdefault(re.sub(r'\d+', '#', d), []).append(d)

        for pattern, members in groups.items():
            for d in members[:max_show]:
                print(f"{ind}{d}/")
                walk(os.path.join(p, d), depth + 1)
            if len(members) > max_show:
                print(f"{ind}... {len(members) - max_show} more matching '{pattern}'")

        for f in files[:max_show]:
            print(f"{ind}{f}")
        if len(files) > max_show:
            print(f"{ind}... {len(files) - max_show} more files")

    print(f"{os.path.basename(path.rstrip('/'))}/")
    walk(path, 1)

explore("CheXpert-v1.0-small/CheXpert-v1.0-small")

CheXpert-v1.0-small/
  train/
    patient00001/
      study1/
        view1_frontal.jpg
    patient00002/
      study1/
        view1_frontal.jpg
        view2_lateral.jpg
      study2/
        view1_frontal.jpg
    patient00003/
      study1/
        view1_frontal.jpg
    ... 64537 more matching 'patient#'
  valid/
    patient64541/
      study1/
        view1_frontal.jpg
    patient64542/
      study1/
        view1_frontal.jpg
        view2_lateral.jpg
    patient64543/
      study1/
        view1_frontal.jpg
    ... 197 more matching 'patient#'
  train.csv
  valid.csv


In [177]:
import pandas as pd
df = pd.read_csv("CheXpert-v1.0-small/CheXpert-v1.0-small/train.csv")
print(df.shape)
df.head()

(223414, 19)


,Path,Sex,Age,Frontal/Lateral,AP/PA,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices
0,CheXpert-v1.0-small/train/patient00001/study1/...,Female,68,Frontal,AP,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,1.0
1,CheXpert-v1.0-small/train/patient00002/study2/...,Female,87,Frontal,AP,NaN,NaN,-1.0,1.0,NaN,-1.0,-1.0,NaN,-1.0,NaN,-1.0,NaN,1.0,NaN
2,CheXpert-v1.0-small/train/patient00002/study1/...,Female,83,Frontal,AP,NaN,NaN,NaN,1.0,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN
3,CheXpert-v1.0-small/train/patient00002/study1/...,Female,83,Lateral,NaN,NaN,NaN,NaN,1.0,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN
4,CheXpert-v1.0-small/train/patient00003/study1/...,Male,41,Frontal,AP,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN


In [178]:
df['AP/PA'].value_counts(dropna=False)

AP/PA
AP     161590
NaN     32387
PA      29420
LL         16
RL          1
Name: count, dtype: int64

In [179]:
df.columns

Index(['Path', 'Sex', 'Age', 'Frontal/Lateral', 'AP/PA', 'No Finding',
       'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
       'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis',
       'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture',
       'Support Devices'],
      dtype='str')

In [180]:
df1 = df[~(df[df.columns[5:]]==-1).any(axis='columns')]

In [181]:
df.shape

(223414, 19)

In [182]:
df1.shape

(138358, 19)

In [183]:
label_cols = df1.columns[5:]

In [184]:
df1[label_cols] = df1[label_cols].fillna(0)

In [185]:
df1[label_cols].isnull().sum().sum()

np.int64(0)

In [186]:
df[label_cols].isnull().sum().sum()

np.int64(2277974)

In [187]:
label_cols

Index(['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly',
       'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia',
       'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other',
       'Fracture', 'Support Devices'],
      dtype='str')

In [188]:
df1 = df1[df1["Frontal/Lateral"] == 'Frontal']

In [189]:
df1.shape

(118286, 19)

- Removed uncertain labels (-1)
- Filled missing labels (NaN → 0)
- Kept frontal views only

118k clean, frontal chest X-rays with 14 binary labels. That's your training data.

In [190]:
import torch
from open_clip import create_model_from_pretrained, get_tokenizer

print(torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

2.13.0
MPS available: True


In [191]:
import torch
from open_clip import create_model_from_pretrained, get_tokenizer

model, preprocess = create_model_from_pretrained(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
)
tokenizer = get_tokenizer(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
)
model = model.eval()
print("loaded")

loaded


In [194]:
from PIL import Image
import torch

concepts = [
    "blunted costophrenic angle",
    "enlarged cardiac silhouette",
    "normal chest radiograph",
    "a chest x-ray showing blunted costophrenic angle",
    "My nose is itching"
]

img = Image.open("CheXpert-v1.0-small/CheXpert-v1.0-small/train/patient00001/study1/view1_frontal.jpg").convert("RGB")
img_t = preprocess(img).unsqueeze(0)
txt_t = tokenizer(concepts, context_length=256)

with torch.no_grad():
    img_emb, txt_emb, _ = model(img_t, txt_t)
    scores = (img_emb @ txt_emb.T).squeeze()

for c, s in zip(concepts, scores):
    print(f"{c}: {s:.4f}")

blunted costophrenic angle: 0.3825
enlarged cardiac silhouette: 0.3612
normal chest radiograph: 0.3752
a chest x-ray showing blunted costophrenic angle: 0.4367
My nose is itching: 0.1430


In [195]:
import os

print("HF_HUB_DISABLE_XET:", os.environ.get("HF_HUB_DISABLE_XET"))
print("TRANSFORMERS_OFFLINE:", os.environ.get("TRANSFORMERS_OFFLINE"))
print("HF_HUB_OFFLINE:", os.environ.get("HF_HUB_OFFLINE"))

HF_HUB_DISABLE_XET: None
TRANSFORMERS_OFFLINE: None
HF_HUB_OFFLINE: None


In [196]:

# ============================================================
# Cell 1 — imports + config
# ============================================================
import json
import torch
import open_clip
 
MAX_LEN = 50
SIM_TO_LABEL_THRESH = 0.9
SIM_TO_CONCEPT_THRESH = 0.9
 # SIM_TO_LABEL_THRESH keeps the bottleneck from trivially encoding the label itself
 # (e.g. a concept literally named "cardiomegaly" would make Cardiomegaly prediction circular)
 # SIM_TO_CONCEPT_THRESH removes near-duplicate concepts so the linear probe isn't
 # given multiple near-identical columns competing for the same signal
 
CLASS_LABELS = [
    "No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
    "Lung Lesion", "Edema", "Consolidation", "Pneumonia", "Atelectasis",
    "Pneumothorax", "Pleural Effusion", "Pleural Other", "Fracture", "Support Devices",
]
 

In [198]:
# ============================================================
# Cell 2 — load raw concepts, flatten, dedup exact strings
# ============================================================
raw = json.load(open("../data/raw_concepts.json"))
 
seen = set()
candidates = []
for pathology, concepts in raw.items():
    for c in concepts:
        if c.strip().lower() not in seen:
            seen.add(c.strip().lower())
            candidates.append(c)
 
print(f"Flattened + exact-deduped: {len(candidates)} candidates")
 

Flattened + exact-deduped: 107 candidates


In [199]:
 
# ============================================================
# Cell 3 — length filter
# ============================================================
candidates = [c for c in candidates if len(c) <= MAX_LEN]
print(f"After length filter: {len(candidates)}")
 

After length filter: 107


In [200]:
# ============================================================
# Cell 4 — load BiomedCLIP (reuse your already-verified model/tokenizer
# if they're still in memory from your smoke test — no need to reload)
# ============================================================
model, _, preprocess = open_clip.create_model_and_transforms(
    "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
)
tokenizer = open_clip.get_tokenizer(
    "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
)
model.eval()

CustomTextCLIP(
  (visual): TimmModel(
    (trunk): VisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
        (norm): Identity()
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (patch_drop): Identity()
      (norm_pre): Identity()
      (blocks): Sequential(
        (0): Block(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Attention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (proj): Linear(in_features=768, out_features=768, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): Identity()
          (drop_path1): Identity()
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          

In [201]:
# ============================================================
# Cell 5 — encode labels and candidates (one function, used twice)
# ============================================================
def encode(strings):
    toks = tokenizer(strings, context_length=256)
    with torch.no_grad():
        emb = model.encode_text(toks)
    return emb / emb.norm(dim=-1, keepdim=True)
 
label_emb = encode(CLASS_LABELS)
concept_emb = encode(candidates)

In [202]:
# ============================================================
# Cell 6 — filter: drop concepts too similar to any class label
# ============================================================
# Dropping concepts too similar to a label name prevents the CBM from trivially
# "cheating" by encoding the label as a concept, rather than a genuine radiological finding
max_sim_to_label = (concept_emb @ label_emb.T).max(dim=1).values
keep = max_sim_to_label <= SIM_TO_LABEL_THRESH
 
candidates = [c for c, k in zip(candidates, keep.tolist()) if k]
concept_emb = concept_emb[keep]
print(f"After label-similarity filter: {len(candidates)}")

After label-similarity filter: 105


In [204]:
# ============================================================
# Cell 7 — filter: drop near-duplicate concepts (greedy, keep first)
# ============================================================
# Greedy dedup, not exhaustive clustering: keeps the first-seen concept in near-duplicate
# groups, which is simple and deterministic but sensitive to input ordering — fine here
# since candidates come from a fixed seed list, not a randomly-ordered source
sim = concept_emb @ concept_emb.T
dropped = set()
final = []
 
for i in range(len(candidates)):
    if i in dropped:
        continue
    final.append(candidates[i])
    for j in range(i + 1, len(candidates)):
        if j not in dropped and sim[i, j] > SIM_TO_CONCEPT_THRESH:
            dropped.add(j)
 
print(f"After concept-concept similarity filter: {len(final)}")

After concept-concept similarity filter: 95


In [205]:
# ============================================================
# Cell 8 — save
# ============================================================
json.dump(final, open("concepts_stage1.json", "w"), indent=2)
print("Saved concepts_stage1.json")
final  # display in notebook to eyeball the survivors

Saved concepts_stage1.json


['clear lung fields',
 'normal cardiac silhouette',
 'sharp costophrenic angles',
 'normal mediastinal contour',
 'no focal opacity',
 'normal bony thorax',
 'symmetric lung inflation',
 'unremarkable chest radiograph',
 'widened mediastinal silhouette',
 'enlarged cardiac contour',
 'widened superior mediastinum',
 'mediastinal mass effect',
 'abnormal aortic contour',
 'tracheal deviation',
 'widened vascular pedicle',
 'loss of mediastinal margins',
 'increased cardiothoracic ratio',
 'globular heart shape',
 'left ventricular enlargement',
 'prominent cardiac apex',
 'cardiac border exceeds half thorax width',
 'biventricular enlargement',
 'boot shaped heart',
 'hazy lung opacity',
 'increased lung density',
 'airspace opacification',
 'patchy parenchymal opacity',
 'obscured vascular markings',
 'ill defined opacity',
 'diffuse ground glass opacity',
 'focal consolidative opacity',
 'solitary pulmonary nodule',
 'well circumscribed mass',
 'cavitary lung lesion',
 'spiculated nod

In [206]:
# ============================================================
# Cell 1 — imports + config
# ============================================================
import pandas as pd
import numpy as np
 
np.random.seed(42)
 
# Flip this to False later to skip subsampling entirely and run on all 118k rows.
# Everything downstream (Cell 4 onward) works unchanged either way.
USE_SUBSAMPLE = False
TARGET_ROWS = 10_000
 
LABEL_COLS = [
    "No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
    "Lung Lesion", "Edema", "Consolidation", "Pneumonia", "Atelectasis",
    "Pneumothorax", "Pleural Effusion", "Pleural Other", "Fracture", "Support Devices",
]
 

In [207]:
df1.shape

(118286, 19)

In [208]:
df1.head(4)

,Path,Sex,Age,Frontal/Lateral,AP/PA,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices
0,CheXpert-v1.0-small/train/patient00001/study1/...,Female,68,Frontal,AP,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,CheXpert-v1.0-small/train/patient00003/study1/...,Male,41,Frontal,AP,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,CheXpert-v1.0-small/train/patient00004/study1/...,Female,20,Frontal,PA,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,CheXpert-v1.0-small/train/patient00005/study1/...,Male,33,Frontal,PA,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [209]:
# ============================================================
# Cell 2 — use df1 (already filtered: no -1s, NaN→0, frontal only)
# If df1 is not in memory, replace this line with:
#   df = pd.read_csv("your_filtered_file.csv")
# ============================================================
df = df1.copy()
df["patient_id"] = df["Path"].str.extract(r"(patient\d+)")
 
n_patients = df["patient_id"].nunique()
print(f"Rows: {len(df)}, unique patients: {n_patients}")
 

Rows: 118286, unique patients: 49404


In [210]:
# ============================================================
# Cell 3 — SUBSAMPLING (only runs if USE_SUBSAMPLE is True)
# Skip/ignore this cell's effect entirely when USE_SUBSAMPLE = False —
# `sub_df` just becomes the full `df` below.
# ============================================================
if USE_SUBSAMPLE:
    patient_labels = df.groupby("patient_id")[LABEL_COLS].max()
    label_prevalence = patient_labels[LABEL_COLS].sum().sort_values()
 
    def rarest_label(row):
        positives = [l for l in LABEL_COLS if row[l] == 1]
        if not positives:
            return "none"
        return min(positives, key=lambda l: label_prevalence[l])
 
    patient_labels["strat_key"] = patient_labels.apply(rarest_label, axis=1)
 
    rows_per_patient = len(df) / n_patients
    target_patients = int(TARGET_ROWS / rows_per_patient)
 
    sampled_patients = (
        patient_labels
        .groupby("strat_key", group_keys=False)
        .apply(lambda g: g.sample(
            n=max(1, round(len(g) * target_patients / n_patients)),
            random_state=42
        ))
        .index
    )
    sampled_patients = pd.Index(sampled_patients).unique()
 
    sub_df = df[df["patient_id"].isin(sampled_patients)].copy()
    print(f"Subsampled: {len(sampled_patients)} patients, {len(sub_df)} rows")
else:
    sub_df = df
    print(f"Using full dataset: {n_patients} patients, {len(sub_df)} rows")
 

Using full dataset: 49404 patients, 118286 rows


In [211]:
# ============================================================
# Cell 4 — split PATIENTS 80/10/10, then assign rows by patient
# (identical whether sub_df is the subsample or the full dataframe)
# ============================================================
patients = sub_df["patient_id"].unique().astype(str)  # force plain numpy array
np.random.shuffle(patients)
n = len(patients)
n_train = int(0.8 * n)
n_val = int(0.1 * n)
 
train_patients = set(patients[:n_train])
val_patients = set(patients[n_train:n_train + n_val])
test_patients = set(patients[n_train + n_val:])
 
train_df = sub_df[sub_df["patient_id"].isin(train_patients)]
val_df = sub_df[sub_df["patient_id"].isin(val_patients)]
test_df = sub_df[sub_df["patient_id"].isin(test_patients)]
 
print(f"Patients -> train {len(train_patients)}, val {len(val_patients)}, test {len(test_patients)}")
print(f"Rows     -> train {len(train_df)}, val {len(val_df)}, test {len(test_df)}")
 
assert not (train_patients & val_patients)
assert not (train_patients & test_patients)
assert not (val_patients & test_patients)
print("No patient overlap across splits — confirmed.")
 

/var/folders/1f/h4qhh2cx0c1fc6x1f1v148540000gn/T/ipykernel_58117/75565785.py:6: UserWarning: you are shuffling a 'ArrowStringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  np.random.shuffle(patients)


Patients -> train 39523, val 4940, test 4941
Rows     -> train 94563, val 11925, test 11798
No patient overlap across splits — confirmed.


In [212]:
# ============================================================
# Cell 5 — save (filename suffix follows the toggle automatically)
# ============================================================
suffix = "_sub" if USE_SUBSAMPLE else "_full"
 
train_df.to_csv(f"train{suffix}.csv", index=False)
val_df.to_csv(f"val{suffix}.csv", index=False)
test_df.to_csv(f"test{suffix}.csv", index=False)
print(f"Saved train{suffix}.csv, val{suffix}.csv, test{suffix}.csv")
 
for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"\n{name} positive label counts:")
    print(split[LABEL_COLS].sum().sort_values())

Saved train_full.csv, val_full.csv, test_full.csv

train positive label counts:
Pleural Other                  1117.0
Pneumonia                      2388.0
Lung Lesion                    3465.0
Fracture                       4021.0
Enlarged Cardiomediastinum     4790.0
Consolidation                  6217.0
Pneumothorax                  10884.0
Cardiomegaly                  12399.0
No Finding                    13553.0
Atelectasis                   16969.0
Edema                         27468.0
Lung Opacity                  38598.0
Pleural Effusion              40005.0
Support Devices               54648.0
dtype: float64

val positive label counts:
Pleural Other                  163.0
Pneumonia                      312.0
Fracture                       450.0
Lung Lesion                    463.0
Enlarged Cardiomediastinum     617.0
Consolidation                  777.0
Cardiomegaly                  1501.0
Pneumothorax                  1553.0
No Finding                    1680.0
Atelectasis 

# Step 3

In [213]:
# ============================================================
# Cell 1 — load concepts (order is fixed from here on, never reshuffle)
# ============================================================
import json
import torch
 
concepts = json.load(open("concepts_stage1.json"))
print(f"Loaded {len(concepts)} concepts")
print(f"First 3: {concepts[:3]}")
print(f"Last 3:  {concepts[-3:]}")

Loaded 95 concepts
First 3: ['clear lung fields', 'normal cardiac silhouette', 'sharp costophrenic angles']
Last 3:  ['sternotomy wires', 'picc line', 'tracheostomy tube']


In [214]:
# ============================================================
# Cell 2 — encode through BiomedCLIP text tower
# (model + tokenizer assumed live in kernel from earlier steps)
# ============================================================
txt_tokens = tokenizer(concepts, context_length=256)
 
with torch.no_grad():
    text_emb = model.encode_text(txt_tokens)
    text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)  # L2 normalize
 
print(f"text_emb shape: {text_emb.shape}")   # expect [95, 512]
print(f"Sample norms (should all be ~1.0): {text_emb.norm(dim=-1)[:5]}")

text_emb shape: torch.Size([95, 512])
Sample norms (should all be ~1.0): tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [215]:
# ============================================================
# Cell 3 — save to disk (computed once, never recomputed)
# ============================================================
torch.save(text_emb, "text_emb.pt")
print("Saved text_emb.pt")

Saved text_emb.pt


In [216]:
# ============================================================
# Cell 4 — sanity check: reload and verify
# ============================================================
loaded = torch.load("text_emb.pt")
assert loaded.shape == text_emb.shape
assert torch.allclose(loaded, text_emb)
print(f"Reload check passed — shape {loaded.shape}, values match ✓")
 

Reload check passed — shape torch.Size([95, 512]), values match ✓


### Step 4 - Get the tensor for the imaegs that is the timage embeddigns [image_count, 512]

In [217]:

# ============================================================
# Cell 1 — imports + config
# ============================================================
import numpy as np
import pandas as pd
import torch
from PIL import Image
from pathlib import Path
from tqdm import tqdm
 
DEVICE = "mps"       # Apple Silicon — change to "cuda" or "cpu" if needed
BATCH_SIZE = 32
CHECKPOINT_EVERY = 1000
IMG_ROOT = Path("CheXpert-v1.0-small")   # folder containing train/patient.../...jpg
 
# model + preprocess assumed live in kernel from earlier steps
# CRITICAL: move the model itself to MPS — inputs alone being on MPS
# isn't enough, the weights must be on the same device or PyTorch errors.
model = model.to(DEVICE)

In [219]:
# ============================================================
# Cell 2 — encoding function (shared across all three splits)
# ============================================================
def encode_split(df, split_name):
    """
    Encodes all images in df (in CSV row order) through BiomedCLIP image tower.
    Saves checkpoints every CHECKPOINT_EVERY images to avoid losing progress.
    Returns full embedding array of shape [len(df), 512].
    """
    out_path = Path(f"img_emb_{split_name}.npy")
    ckpt_path = Path(f"img_emb_{split_name}_ckpt.npy")
 
    paths = df["Path"].tolist()
    n = len(paths)
 
    # resume from checkpoint if one exists
    if ckpt_path.exists():
        done = np.load(ckpt_path)
        start_idx = len(done)
        print(f"[{split_name}] Resuming from checkpoint at image {start_idx}/{n}")
    else:
        done = np.zeros((0, 512), dtype=np.float32)
        start_idx = 0
        print(f"[{split_name}] Starting fresh — {n} images to encode")
 
    all_embs = list(done) if len(done) > 0 else []
 
    model.eval()
    batch_imgs = []
    batch_indices = []
 
    def flush_batch():
        if not batch_imgs:
            return
        tensor = torch.stack(batch_imgs).to(DEVICE)
        with torch.no_grad():
            emb = model.encode_image(tensor)
            emb = emb / emb.norm(dim=-1, keepdim=True)  # L2 normalize
        all_embs.extend(emb.cpu().numpy())
        batch_imgs.clear()
        batch_indices.clear()
 
    for i in tqdm(range(start_idx, n), desc=split_name):
        img_path = IMG_ROOT / paths[i]
        img = Image.open(img_path).convert("RGB")
        batch_imgs.append(preprocess(img))
        batch_indices.append(i)
 
        # flush when batch is full
        if len(batch_imgs) == BATCH_SIZE:
            flush_batch()
 
        # checkpoint
        if (i + 1) % CHECKPOINT_EVERY == 0:
            np.save(ckpt_path, np.array(all_embs, dtype=np.float32))
            print(f"[{split_name}] Checkpoint saved at {i + 1}/{n}")
 
    # flush any remaining images in the last partial batch
    flush_batch()
 
    # save final output, remove checkpoint
    final = np.array(all_embs, dtype=np.float32)
    np.save(out_path, final)
    if ckpt_path.exists():
        ckpt_path.unlink()
 
    print(f"[{split_name}] Done — saved {out_path}, shape {final.shape}")
    return final
 

In [221]:
# ============================================================
# Cell 3 — encode all three splits
# (order: train first, then val, then test — each is independent)
# ============================================================
train_df = pd.read_csv("train_full.csv")
val_df   = pd.read_csv("val_full.csv")
test_df  = pd.read_csv("test_full.csv")
 
img_emb_train = encode_split(train_df, "train")
img_emb_val   = encode_split(val_df,   "val")
img_emb_test  = encode_split(test_df,  "test")

[train] Starting fresh — 94563 images to encode




train:   0%|                                         | 0/94563 [00:00<?, ?it/s]

train:   0%|                                | 32/94563 [00:00<18:20, 85.91it/s]

train:   0%|                                | 64/94563 [00:00<16:29, 95.49it/s]

train:   0%|                                | 96/94563 [00:00<15:53, 99.07it/s]

train:   0%|                              | 128/94563 [00:01<15:34, 101.03it/s]

train:   0%|                              | 160/94563 [00:01<15:24, 102.10it/s]

train:   0%|                              | 192/94563 [00:01<15:17, 102.81it/s]

train:   0%|                              | 224/94563 [00:02<15:15, 103.04it/s]

train:   0%|                              | 256/94563 [00:02<15:11, 103.49it/s]

train:   0%|                              | 288/94563 [00:02<15:10, 103.50it/s]

train:   0%|                              | 320/94563 [00:03<15:09, 103.64it/s]

train:   0%|                              | 352/94563 [00:03<15:14, 103.01it/s]

train:   0%|              

[train] Checkpoint saved at 1000/94563




train:   1%|▎                            | 1024/94563 [00:09<14:57, 104.19it/s]

train:   1%|▎                            | 1056/94563 [00:10<14:56, 104.30it/s]

train:   1%|▎                            | 1088/94563 [00:10<14:56, 104.30it/s]

train:   1%|▎                            | 1120/94563 [00:10<14:56, 104.19it/s]

train:   1%|▎                            | 1152/94563 [00:11<14:55, 104.29it/s]

train:   1%|▎                            | 1184/94563 [00:11<14:54, 104.34it/s]

train:   1%|▎                            | 1216/94563 [00:11<14:55, 104.26it/s]

train:   1%|▍                            | 1248/94563 [00:12<14:55, 104.16it/s]

train:   1%|▍                            | 1280/94563 [00:12<14:56, 104.05it/s]

train:   1%|▍                            | 1312/94563 [00:12<14:56, 104.03it/s]

train:   1%|▍                            | 1344/94563 [00:13<14:54, 104.21it/s]

train:   1%|▍                            | 1376/94563 [00:13<14:53, 104.25it/s]

train:   1%|▍             

[train] Checkpoint saved at 2000/94563


...

[test] Checkpoint saved at 10000/11798




test:  85%|█████████████████████████▍    | 10016/11798 [01:40<00:17, 99.50it/s]

test:  85%|█████████████████████████▌    | 10048/11798 [01:40<00:17, 99.55it/s]

test:  85%|█████████████████████████▋    | 10080/11798 [01:41<00:17, 99.63it/s]

test:  86%|█████████████████████████▋    | 10112/11798 [01:41<00:16, 99.39it/s]

test:  86%|█████████████████████████▊    | 10144/11798 [01:41<00:16, 99.53it/s]

test:  86%|█████████████████████████▉    | 10176/11798 [01:42<00:16, 99.48it/s]

test:  87%|█████████████████████████▉    | 10208/11798 [01:42<00:16, 99.06it/s]

test:  87%|██████████████████████████    | 10240/11798 [01:42<00:15, 98.96it/s]

test:  87%|██████████████████████████    | 10272/11798 [01:43<00:15, 98.98it/s]

test:  87%|██████████████████████████▏   | 10304/11798 [01:43<00:14, 99.60it/s]

test:  88%|██████████████████████████▎   | 10336/11798 [01:43<00:14, 99.86it/s]

test:  88%|█████████████████████████▍   | 10368/11798 [01:44<00:14, 100.31it/s]

test:  88%|███████████████

[test] Checkpoint saved at 11000/11798




test:  93%|███████████████████████████▉  | 11008/11798 [01:50<00:08, 97.89it/s]

test:  94%|████████████████████████████  | 11040/11798 [01:50<00:07, 97.99it/s]

test:  94%|████████████████████████████▏ | 11072/11798 [01:51<00:07, 98.42it/s]

test:  94%|████████████████████████████▏ | 11104/11798 [01:51<00:07, 98.60it/s]

test:  94%|████████████████████████████▎ | 11136/11798 [01:51<00:06, 98.77it/s]

test:  95%|████████████████████████████▍ | 11168/11798 [01:52<00:06, 99.02it/s]

test:  95%|████████████████████████████▍ | 11200/11798 [01:52<00:06, 99.16it/s]

test:  95%|████████████████████████████▌ | 11232/11798 [01:52<00:05, 99.19it/s]

test:  95%|████████████████████████████▋ | 11264/11798 [01:53<00:05, 99.17it/s]

test:  96%|████████████████████████████▋ | 11296/11798 [01:53<00:05, 99.13it/s]

test:  96%|████████████████████████████▊ | 11328/11798 [01:53<00:04, 98.98it/s]

test:  96%|████████████████████████████▉ | 11360/11798 [01:54<00:04, 98.87it/s]

test:  97%|███████████████

[test] Done — saved img_emb_test.npy, shape (11798, 512)


## Step - 5 Use the image emeddings to cclacluate the concept simliarity scores

In [222]:

# ============================================================
# Cell 1 — imports + config
# ============================================================
import json
import numpy as np
import torch
from sklearn.cluster import KMeans
from pathlib import Path
 
DEVICE = "mps"
MIN_MAX_SIM = 0.2          # filter 3 threshold
N_CONCEPTS_BEFORE = None   # filled in after loading
 
concepts = json.load(open("concepts_stage1.json"))
text_emb = torch.load("text_emb.pt").to(DEVICE)   # [95, 512]
 
print(f"Concepts loaded: {len(concepts)}")
print(f"text_emb shape: {text_emb.shape}")

Concepts loaded: 95
text_emb shape: torch.Size([95, 512])


In [223]:
# ============================================================
# Cell 2 — helper: compute raw score matrix on MPS
# ============================================================
def compute_scores(img_emb_path):
    """
    Loads image embeddings from disk, moves to MPS,
    computes cosine similarity against all concepts.
    Returns raw scores as numpy array [N, n_concepts].
    """
    img_emb = np.load(img_emb_path).astype(np.float32)
    img_tensor = torch.from_numpy(img_emb).to(DEVICE)   # [N, 512]
    with torch.no_grad():
        scores = img_tensor @ text_emb.T                 # [N, 95] — MPS parallel
    return scores.cpu().numpy()                          # back to CPU for sklearn
 


In [224]:
# ============================================================
# Cell 3 — compute scores for all three splits
# ============================================================
scores_train = compute_scores("img_emb_train.npy")
scores_val   = compute_scores("img_emb_val.npy")
scores_test  = compute_scores("img_emb_test.npy")
 
print(f"scores_train: {scores_train.shape}")   # [7723, 95]
print(f"scores_val:   {scores_val.shape}")     # [1117, 95]
print(f"scores_test:  {scores_test.shape}")    # [977,  95]

scores_train: (94563, 95)
scores_val:   (11925, 95)
scores_test:  (11798, 95)


In [225]:
# ============================================================
# Cell 4 — filter 3: drop concepts with max sim < 0.2
#           computed on train only (val/test must not influence filtering)
# ============================================================
max_sim_per_concept = scores_train.max(axis=0)   # [95] — max across all train images
keep_mask = max_sim_per_concept >= MIN_MAX_SIM
 
concepts_final = [c for c, keep in zip(concepts, keep_mask) if keep]
n_dropped = len(concepts) - len(concepts_final)
 
print(f"\nFilter 3 results:")
print(f"  Concepts before: {len(concepts)}")
print(f"  Dropped (max sim < {MIN_MAX_SIM}): {n_dropped}")
print(f"  Surviving: {len(concepts_final)}")
 
if n_dropped > 0:
    dropped = [c for c, keep in zip(concepts, keep_mask) if not keep]
    print(f"  Dropped concepts: {dropped}")
 
# apply mask to all three splits
scores_train = scores_train[:, keep_mask]
scores_val   = scores_val[:,   keep_mask]
scores_test  = scores_test[:,  keep_mask]
 
# save surviving concept list — canonical from here on
json.dump(concepts_final, open("../data/concepts_final.json", "w"), indent=2)
print(f"\nSaved concepts_final.json ({len(concepts_final)} concepts)")


Filter 3 results:
  Concepts before: 95
  Dropped (max sim < 0.2): 0
  Surviving: 95

Saved concepts_final.json (95 concepts)


In [226]:
# ============================================================
# Cell 5 — save raw scores (needed for diagnostics + inference)
# ============================================================
np.save("concept_scores_train.npy", scores_train)
np.save("concept_scores_val.npy",   scores_val)
np.save("concept_scores_test.npy",  scores_test)
print("Saved raw concept scores for all splits")
 

Saved raw concept scores for all splits


In [227]:
# ============================================================
# Cell 6 — binarize with KMeans per concept column
#           fit on train, apply same centroids to val + test
# ============================================================
n_concepts = len(concepts_final)
 
def binarize(scores, kmeans_models=None, fit=False):
    """
    Binarizes score matrix column by column using 2-means.
    If fit=True, fits KMeans on this data and returns (binary_matrix, models).
    If fit=False, applies pre-fitted models from training split.
    Higher centroid cluster → 1, lower → 0.
    """
    binary = np.zeros_like(scores, dtype=np.int8)
    models = [] if fit else kmeans_models
 
    for j in range(scores.shape[1]):
        col = scores[:, j].reshape(-1, 1)
        if fit:
            km = KMeans(n_clusters=2, random_state=42, n_init=10)
            km.fit(col)
            models.append(km)
        else:
            km = kmeans_models[j]
 
        labels = km.predict(col)
        # figure out which cluster label corresponds to the higher centroid
        c0, c1 = km.cluster_centers_[0][0], km.cluster_centers_[1][0]
        high_label = 0 if c0 > c1 else 1
        binary[:, j] = (labels == high_label).astype(np.int8)
 
    if fit:
        return binary, models
    return binary
 
# fit on train, transform val + test with the same models
concept_matrix_train, kmeans_models = binarize(scores_train, fit=True)
concept_matrix_val  = binarize(scores_val,  kmeans_models=kmeans_models)
concept_matrix_test = binarize(scores_test, kmeans_models=kmeans_models)
 
print(f"\nconcept_matrix_train: {concept_matrix_train.shape}")
print(f"concept_matrix_val:   {concept_matrix_val.shape}")
print(f"concept_matrix_test:  {concept_matrix_test.shape}")
 


concept_matrix_train: (94563, 95)
concept_matrix_val:   (11925, 95)
concept_matrix_test:  (11798, 95)


In [228]:
# ============================================================
# Cell 7 — save binarized matrices
# ============================================================
np.save("concept_matrix_train.npy", concept_matrix_train)
np.save("concept_matrix_val.npy",   concept_matrix_val)
np.save("concept_matrix_test.npy",  concept_matrix_test)
print("Saved binarized concept matrices for all splits")

 

Saved binarized concept matrices for all splits


In [229]:
# ============================================================
# Cell 8 — sanity checks
# ============================================================
# centroid separation per concept — flag anything suspiciously close
print("\nCentroid separation per concept (flag if < 0.05):")
warnings = []
for j, (km, concept) in enumerate(zip(kmeans_models, concepts_final)):
    sep = abs(km.cluster_centers_[0][0] - km.cluster_centers_[1][0])
    if sep < 0.05:
        warnings.append((concept, sep))
 
if warnings:
    print(f"  {len(warnings)} concepts with low separation:")
    for c, s in warnings:
        print(f"    '{c}': {s:.4f}")
else:
    print("  All concepts have healthy centroid separation ✓")
 
# activation rate per concept in train (should not be all 0s or all 1s)
activation_rate = concept_matrix_train.mean(axis=0)
print(f"\nActivation rate stats across concepts (train):")
print(f"  min: {activation_rate.min():.3f}")
print(f"  max: {activation_rate.max():.3f}")
print(f"  mean: {activation_rate.mean():.3f}")
degenerate = [(concepts_final[j], activation_rate[j])
              for j in range(n_concepts)
              if activation_rate[j] < 0.01 or activation_rate[j] > 0.99]
if degenerate:
    print(f"  Degenerate concepts (nearly all 0 or all 1): {degenerate}")
else:
    print("  No degenerate concepts ✓")


Centroid separation per concept (flag if < 0.05):
  32 concepts with low separation:
    'clear lung fields': 0.0429
    'symmetric lung inflation': 0.0435
    'widened vascular pedicle': 0.0489
    'loss of mediastinal margins': 0.0464
    'hazy lung opacity': 0.0448
    'increased lung density': 0.0408
    'airspace opacification': 0.0435
    'patchy parenchymal opacity': 0.0483
    'diffuse ground glass opacity': 0.0481
    'focal consolidative opacity': 0.0470
    'irregular lung mass': 0.0480
    'interstitial lung markings': 0.0436
    'kerley b lines': 0.0493
    'peribronchial cuffing': 0.0443
    'vascular redistribution': 0.0411
    'fluid in fissures': 0.0453
    'segmental consolidation': 0.0475
    'unilateral lung infiltrate': 0.0481
    'volume loss in lung': 0.0440
    'shifted fissure': 0.0421
    'platelike atelectasis': 0.0492
    'collapsed lung segment': 0.0345
    'visceral pleural line': 0.0443
    'deep sulcus sign': 0.0494
    'small apical lucency': 0.0440
  

## Step 7

In [230]:
# ============================================================
# Cell 1 — imports + config
# ============================================================
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score

DEVICE = "mps"
L1_LAMBDA = 1e-4     # encourages sparse weights
N_EPOCHS = 100
LR = 1e-3

In [232]:
# ============================================================
# Cell 2 — load labels from the same CSVs used for encoding
# These are the exact row-order source encode_split() read from,
# so row i here == row i of img_emb_{split}.npy
# ============================================================
train_df = pd.read_csv("train_full.csv")
val_df   = pd.read_csv("val_full.csv")

y_train = train_df[LABEL_COLS].values
y_val   = val_df[LABEL_COLS].values

print(f"y_train: {y_train.shape}, y_val: {y_val.shape}")

y_train: (94563, 14), y_val: (11925, 14)


In [247]:
train_df.head(-3)

,Path,Sex,Age,Frontal/Lateral,AP/PA,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices,patient_id
0,CheXpert-v1.0-small/train/patient00001/study1/...,Female,68,Frontal,AP,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,patient00001
1,CheXpert-v1.0-small/train/patient00003/study1/...,Male,41,Frontal,AP,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,patient00003
2,CheXpert-v1.0-small/train/patient00005/study1/...,Male,33,Frontal,PA,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,patient00005
3,CheXpert-v1.0-small/train/patient00005/study2/...,Male,33,Frontal,AP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,patient00005
4,CheXpert-v1.0-small/train/patient00005/study2/...,Male,33,Frontal,AP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,patient00005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94555,CheXpert-v1.0-small/train/patient64522/study1/...,Female,21,Frontal,AP,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,patient64522
94556,CheXpert-v1.0-small/train/patient64524/study1/...,Female,61,Frontal,AP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,patient64524
94557,CheXpert-v1.0-small/train/patient64527/study2/...,Male,85,Frontal,AP,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,patient64527
94558,CheXpert-v1.0-small/train/patient64529/study1/...,Male,81,Frontal,AP,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,patient64529


In [248]:
df1.head(-3)

,Path,Sex,Age,Frontal/Lateral,AP/PA,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices
0,CheXpert-v1.0-small/train/patient00001/study1/...,Female,68,Frontal,AP,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,CheXpert-v1.0-small/train/patient00003/study1/...,Male,41,Frontal,AP,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,CheXpert-v1.0-small/train/patient00004/study1/...,Female,20,Frontal,PA,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,CheXpert-v1.0-small/train/patient00005/study1/...,Male,33,Frontal,PA,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
9,CheXpert-v1.0-small/train/patient00005/study2/...,Male,33,Frontal,AP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223396,CheXpert-v1.0-small/train/patient64527/study2/...,Male,85,Frontal,AP,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
223399,CheXpert-v1.0-small/train/patient64529/study1/...,Male,81,Frontal,AP,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
223400,CheXpert-v1.0-small/train/patient64530/study1/...,Male,65,Frontal,AP,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
223402,CheXpert-v1.0-small/train/patient64532/study1/...,Female,52,Frontal,AP,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [233]:
# ============================================================
# Cell 3 — load onto MPS
# y_train, y_val must already exist as [N, 14] numpy arrays,
# same row order as concept_matrix_train / concept_matrix_val
# ============================================================
X_train = torch.tensor(concept_matrix_train, dtype=torch.float32).to(DEVICE)
X_val   = torch.tensor(concept_matrix_val,   dtype=torch.float32).to(DEVICE)
y_train_t = torch.tensor(y_train, dtype=torch.float32).to(DEVICE)
y_val_t   = torch.tensor(y_val,   dtype=torch.float32).to(DEVICE)

n_concepts = X_train.shape[1]
n_labels = y_train_t.shape[1]
print(f"X_train: {X_train.shape}, y_train: {y_train_t.shape}")

X_train: torch.Size([94563, 95]), y_train: torch.Size([94563, 14])


In [234]:
# ============================================================
# Cell 4 — model, loss, optimizer
# ============================================================
model = nn.Linear(n_concepts, n_labels).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [235]:
# ============================================================
# Cell 5 — training loop
# ============================================================
best_val_auroc = 0

for epoch in range(N_EPOCHS):
    model.train()
    optimizer.zero_grad()
    logits = model(X_train)
    loss = criterion(logits, y_train_t)
    l1_penalty = sum(p.abs().sum() for p in model.parameters())
    loss = loss + L1_LAMBDA * l1_penalty
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_probs = torch.sigmoid(model(X_val)).cpu().numpy()

    y_val_np = y_val_t.cpu().numpy()
    aurocs = [
        roc_auc_score(y_val_np[:, i], val_probs[:, i])
        for i in range(n_labels)
        if len(np.unique(y_val_np[:, i])) > 1
    ]
    macro_auroc = np.mean(aurocs)

    if macro_auroc > best_val_auroc:
        best_val_auroc = macro_auroc
        torch.save(model.state_dict(), "linear_head_best.pt")

    if epoch % 10 == 0:
        print(f"epoch {epoch:3d}  loss {loss.item():.4f}  val macro AUROC {macro_auroc:.4f}")

epoch   0  loss 0.6942  val macro AUROC 0.4914
epoch  10  loss 0.5632  val macro AUROC 0.5199
epoch  20  loss 0.4839  val macro AUROC 0.5442
epoch  30  loss 0.4394  val macro AUROC 0.5602
epoch  40  loss 0.4147  val macro AUROC 0.5707
epoch  50  loss 0.4004  val macro AUROC 0.5784
epoch  60  loss 0.3913  val macro AUROC 0.5846
epoch  70  loss 0.3850  val macro AUROC 0.5898
epoch  80  loss 0.3805  val macro AUROC 0.5944
epoch  90  loss 0.3770  val macro AUROC 0.5983


In [236]:
# ============================================================
# Cell 6 — report
# ============================================================
print(f"best val macro AUROC: {best_val_auroc:.4f}")

best val macro AUROC: 0.6014


In [238]:
# ============================================================
# Cell 1 — load test data + best checkpoint
# ============================================================
test_df = pd.read_csv("test_full.csv")
y_test = test_df[LABEL_COLS].values

X_test = torch.tensor(concept_matrix_test, dtype=torch.float32).to(DEVICE)
y_test_t = torch.tensor(y_test, dtype=torch.float32).to(DEVICE)

model.load_state_dict(torch.load("linear_head_best.pt"))
model.eval()

print(f"X_test: {X_test.shape}, y_test: {y_test_t.shape}")

X_test: torch.Size([11798, 95]), y_test: torch.Size([11798, 14])


In [239]:
# ============================================================
# Cell 2 — per-label AUROC on test set
# ============================================================
with torch.no_grad():
    test_probs = torch.sigmoid(model(X_test)).cpu().numpy()

y_test_np = y_test_t.cpu().numpy()

print(f"{'Label':<28} {'AUROC':>8} {'n_pos':>7}")
per_label_auroc = {}
for i, name in enumerate(LABEL_COLS):
    if len(np.unique(y_test_np[:, i])) > 1:
        auc = roc_auc_score(y_test_np[:, i], test_probs[:, i])
        per_label_auroc[name] = auc
        print(f"{name:<28} {auc:8.4f} {int(y_test_np[:, i].sum()):7d}")
    else:
        print(f"{name:<28} {'skip':>8} {int(y_test_np[:, i].sum()):7d}  (no positives in test)")

macro_test_auroc = np.mean(list(per_label_auroc.values()))
print(f"\nmacro AUROC (test): {macro_test_auroc:.4f}")

Label                           AUROC   n_pos
No Finding                     0.7104    1662
Enlarged Cardiomediastinum     0.5065     639
Cardiomegaly                   0.5792    1542
Lung Opacity                   0.6906    4921
Lung Lesion                    0.5464     469
Edema                          0.7231    3400
Consolidation                  0.5710     752
Pneumonia                      0.6157     274
Atelectasis                    0.4853    2199
Pneumothorax                   0.5334    1265
Pleural Effusion               0.7882    5084
Pleural Other                  0.5296     147
Fracture                       0.4809     518
Support Devices                0.7619    6857

macro AUROC (test): 0.6087


In [240]:
# ============================================================
# Cell 3 — baseline: plain linear probe on raw CLIP embeddings
# No concept bottleneck at all. This is the accuracy ceiling —
# what you're giving up for interpretability.
# ============================================================
img_emb_train_raw = np.load("img_emb_train.npy")
img_emb_val_raw   = np.load("img_emb_val.npy")
img_emb_test_raw  = np.load("img_emb_test.npy")

Xb_train = torch.tensor(img_emb_train_raw, dtype=torch.float32).to(DEVICE)
Xb_val   = torch.tensor(img_emb_val_raw,   dtype=torch.float32).to(DEVICE)
Xb_test  = torch.tensor(img_emb_test_raw,  dtype=torch.float32).to(DEVICE)

baseline = nn.Linear(Xb_train.shape[1], n_labels).to(DEVICE)
opt_b = torch.optim.Adam(baseline.parameters(), lr=LR)
crit_b = nn.BCEWithLogitsLoss()

best_val_b = 0
for epoch in range(N_EPOCHS):
    baseline.train()
    opt_b.zero_grad()
    loss = crit_b(baseline(Xb_train), y_train_t)
    loss.backward()
    opt_b.step()

    baseline.eval()
    with torch.no_grad():
        val_probs_b = torch.sigmoid(baseline(Xb_val)).cpu().numpy()
    aurocs_b = [roc_auc_score(y_val_t.cpu().numpy()[:, i], val_probs_b[:, i])
                for i in range(n_labels) if len(np.unique(y_val_t.cpu().numpy()[:, i])) > 1]
    macro_b = np.mean(aurocs_b)
    if macro_b > best_val_b:
        best_val_b = macro_b
        torch.save(baseline.state_dict(), "baseline_best.pt")

baseline.load_state_dict(torch.load("baseline_best.pt"))
baseline.eval()
with torch.no_grad():
    test_probs_b = torch.sigmoid(baseline(Xb_test)).cpu().numpy()

aurocs_b_test = [roc_auc_score(y_test_np[:, i], test_probs_b[:, i])
                  for i in range(n_labels) if len(np.unique(y_test_np[:, i])) > 1]
macro_baseline_test = np.mean(aurocs_b_test)

print(f"CBM  macro AUROC (test): {macro_test_auroc:.4f}")
print(f"Baseline (no bottleneck) macro AUROC (test): {macro_baseline_test:.4f}")
print(f"Accuracy cost of the bottleneck: {macro_baseline_test - macro_test_auroc:.4f}")

CBM  macro AUROC (test): 0.6087
Baseline (no bottleneck) macro AUROC (test): 0.6318
Accuracy cost of the bottleneck: 0.0230


In [241]:
# ============================================================
# Cell 4 — faithfulness check
# For each label, find the concept the model itself leans on
# most (largest weight magnitude), then check whether that
# concept is actually MORE active in positive cases than
# negative cases. If it's not, the model's stated reason
# for a prediction doesn't match the data.
# ============================================================
weights = model.weight.detach().cpu().numpy()  # [14, n_concepts]

print(f"{'Label':<28} {'Top concept':<30} {'weight':>7} {'pos rate':>9} {'neg rate':>9}")
for i, label_name in enumerate(LABEL_COLS):
    top_j = np.argmax(np.abs(weights[i]))
    concept_name = concepts_final[top_j]
    w = weights[i, top_j]

    pos_mask = y_train[:, i] == 1
    neg_mask = y_train[:, i] == 0
    pos_rate = concept_matrix_train[pos_mask, top_j].mean() if pos_mask.sum() > 0 else float("nan")
    neg_rate = concept_matrix_train[neg_mask, top_j].mean() if neg_mask.sum() > 0 else float("nan")

    print(f"{label_name:<28} {concept_name:<30} {w:7.3f} {pos_rate:9.3f} {neg_rate:9.3f}")

Label                        Top concept                     weight  pos rate  neg rate
No Finding                   callus formation                -0.158     0.166     0.466
Enlarged Cardiomediastinum   increased cardiothoracic ratio  -0.154     0.500     0.469
Cardiomegaly                 kerley b lines                  -0.136     0.429     0.529
Lung Opacity                 rib cortical discontinuity      -0.150     0.413     0.561
Lung Lesion                  widened mediastinal silhouette  -0.159     0.378     0.621
Edema                        cavitary lung lesion            -0.149     0.349     0.508
Consolidation                pacemaker leads                 -0.145     0.373     0.478
Pneumonia                    boot shaped heart               -0.162     0.348     0.465
Atelectasis                  abnormal aortic contour         -0.140     0.580     0.544
Pneumothorax                 biventricular enlargement       -0.148     0.292     0.409
Pleural Effusion             bat

In [242]:
# ============================================================
# Step 7 — print all results in one place
# ============================================================

# ── per-label AUROC (CBM) ─────────────────────────────────
print("=" * 58)
print("Per-label AUROC — CBM (test set)")
print("=" * 58)
print(f"{'Label':<28} {'AUROC':>8} {'n_pos':>7}")
per_label_auroc = {}
for i, name in enumerate(LABEL_COLS):
    if len(np.unique(y_test_np[:, i])) > 1:
        auc = roc_auc_score(y_test_np[:, i], test_probs[:, i])
        per_label_auroc[name] = auc
        print(f"{name:<28} {auc:8.4f} {int(y_test_np[:, i].sum()):7d}")
    else:
        print(f"{name:<28} {'skip':>8} {int(y_test_np[:, i].sum()):7d}")
macro_cbm = np.mean(list(per_label_auroc.values()))
print(f"\nCBM macro AUROC (test): {macro_cbm:.4f}")

# ── faithfulness ─────────────────────────────────────────
print("\n" + "=" * 58)
print("Faithfulness — top concept per label")
print("=" * 58)
weights = model.weight.detach().cpu().numpy()
print(f"{'Label':<28} {'Top concept':<32} {'w':>6} {'pos%':>7} {'neg%':>7}")
for i, name in enumerate(LABEL_COLS):
    top_j = np.argmax(np.abs(weights[i]))
    w = weights[i, top_j]
    pos_mask = y_train[:, i] == 1
    neg_mask = y_train[:, i] == 0
    pos_r = concept_matrix_train[pos_mask, top_j].mean() if pos_mask.sum() > 0 else float("nan")
    neg_r = concept_matrix_train[neg_mask, top_j].mean() if neg_mask.sum() > 0 else float("nan")
    print(f"{name:<28} {concepts_final[top_j]:<32} {w:6.3f} {pos_r:7.3f} {neg_r:7.3f}")

Per-label AUROC — CBM (test set)
Label                           AUROC   n_pos
No Finding                     0.7104    1662
Enlarged Cardiomediastinum     0.5065     639
Cardiomegaly                   0.5792    1542
Lung Opacity                   0.6906    4921
Lung Lesion                    0.5464     469
Edema                          0.7231    3400
Consolidation                  0.5710     752
Pneumonia                      0.6157     274
Atelectasis                    0.4853    2199
Pneumothorax                   0.5334    1265
Pleural Effusion               0.7882    5084
Pleural Other                  0.5296     147
Fracture                       0.4809     518
Support Devices                0.7619    6857

CBM macro AUROC (test): 0.6087

Faithfulness — top concept per label
Label                        Top concept                           w    pos%    neg%
No Finding                   callus formation                 -0.158   0.166   0.466
Enlarged Cardiomediastinum   increase

In [243]:
# ============================================================
# Step 8 — interpretability output
# show top driving concepts + one intervention per example
# focusing on Pleural Effusion as the strongest case
# ============================================================

TARGET_LABEL = "Pleural Effusion"
label_idx = LABEL_COLS.index(TARGET_LABEL)
TOP_K = 5

weights = model.weight.detach().cpu().numpy()   # [14, n_concepts]
label_weights = weights[label_idx]              # [n_concepts]

# find test cases the model called correctly with high confidence
pred_scores = test_probs[:, label_idx]
true_labels = y_test_np[:, label_idx]

high_conf_pos = np.where((pred_scores > 0.6) & (true_labels == 1))[0]
high_conf_neg = np.where((pred_scores < 0.4) & (true_labels == 0))[0]

print(f"High-confidence correct positives: {len(high_conf_pos)}")
print(f"High-confidence correct negatives: {len(high_conf_neg)}")

def explain(row_idx, split="test"):
    cm = concept_matrix_test if split == "test" else concept_matrix_train
    concept_scores_raw = cm[row_idx]
    contributions = label_weights * concept_scores_raw
    top_indices = np.argsort(np.abs(contributions))[::-1][:TOP_K]

    pred = pred_scores[row_idx]
    true = int(true_labels[row_idx])
    path = test_df.iloc[row_idx]["Path"]

    print(f"\nImage: {path}")
    print(f"True label: {true}  |  Model score: {pred:.3f}")
    print(f"{'Concept':<32} {'active':>7} {'weight':>8} {'contribution':>14}")
    for j in top_indices:
        print(f"{concepts_final[j]:<32} {int(concept_scores_raw[j]):>7} "
              f"{label_weights[j]:>8.3f} {contributions[j]:>14.4f}")

    # intervention: flip the top concept and reobserve
    top_j = top_indices[0]
    intervened = concept_scores_raw.copy()
    intervened[top_j] = 1 - intervened[top_j]   # flip 0→1 or 1→0
    intervened_t = torch.tensor(intervened, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        new_score = torch.sigmoid(model(intervened_t))[label_idx].item()
    print(f"\nIntervention: flip '{concepts_final[top_j]}' "
          f"{int(concept_scores_raw[top_j])} → {1 - int(concept_scores_raw[top_j])}")
    print(f"Score before: {pred:.3f}  →  after: {new_score:.3f}")

# show 3 positive and 2 negative examples
print("\n" + "=" * 65)
print(f"Explanations for {TARGET_LABEL}")
print("=" * 65)
for idx in high_conf_pos[:3]:
    explain(idx)
for idx in high_conf_neg[:2]:
    explain(idx)

High-confidence correct positives: 1999
High-confidence correct negatives: 4294

Explanations for Pleural Effusion

Image: CheXpert-v1.0-small/train/patient00011/study11/view1_frontal.jpg
True label: 1  |  Model score: 0.664
Concept                           active   weight   contribution
chest tube                             1    0.139         0.1391
patchy parenchymal opacity             1   -0.123        -0.1227
subpulmonic effusion                   1    0.122         0.1224
boot shaped heart                      1    0.121         0.1210
normal cardiac silhouette              1   -0.105        -0.1053

Intervention: flip 'chest tube' 1 → 0
Score before: 0.664  →  after: 0.633

Image: CheXpert-v1.0-small/train/patient00100/study2/view1_frontal.jpg
True label: 1  |  Model score: 0.601
Concept                           active   weight   contribution
chest tube                             1    0.139         0.1391
patchy parenchymal opacity             1   -0.123        -0.1227
boot 

In [251]:
print(f"High-confidence correct positives: {len(high_conf_pos)}/ {len(pred_scores)}")

High-confidence correct positives: 1999/ 11798
